# FFT Simulation

This notebook performs two simulations for the FFT report:

1. Compare a manually implemented direct DFT with `numpy.fft.fft`.
2. Compare the approximate operation growth $N^2$ and $N\log_2N$.

The generated images are saved in a `figures` folder for upload to Overleaf.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs('figures', exist_ok=True)

def direct_dft(x):
    """Compute the DFT directly from its definition."""
    x = np.asarray(x, dtype=complex)
    N = len(x)
    n = np.arange(N)
    k = n.reshape((N, 1))
    W = np.exp(-2j * np.pi * k * n / N)
    return W @ x

## Simulation 1 — Direct DFT versus FFT

In [ ]:
x = np.array([1, 2, 3, 4], dtype=float)
X_dft = direct_dft(x)
X_fft = np.fft.fft(x)
error = np.abs(X_dft - X_fft)

comparison = pd.DataFrame({
    'k': np.arange(len(x)),
    'Direct DFT': X_dft,
    'NumPy FFT': X_fft,
    'Absolute error': error
})
display(comparison)
print('Maximum absolute error =', error.max())
print('All results equal within tolerance:', np.allclose(X_dft, X_fft))

In [ ]:
k = np.arange(len(x))
fig, axs = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)

axs[0, 0].stem(k, np.abs(X_dft), basefmt=' ')
axs[0, 0].set_title('Direct DFT magnitude')
axs[0, 0].set_xlabel('Frequency bin k')
axs[0, 0].set_ylabel('|X(k)|')
axs[0, 0].grid(True, alpha=0.3)

axs[0, 1].stem(k, np.abs(X_fft), basefmt=' ')
axs[0, 1].set_title('NumPy FFT magnitude')
axs[0, 1].set_xlabel('Frequency bin k')
axs[0, 1].set_ylabel('|X(k)|')
axs[0, 1].grid(True, alpha=0.3)

axs[1, 0].stem(k, np.angle(X_dft), basefmt=' ')
axs[1, 0].set_title('Direct DFT phase')
axs[1, 0].set_xlabel('Frequency bin k')
axs[1, 0].set_ylabel('Phase (rad)')
axs[1, 0].grid(True, alpha=0.3)

axs[1, 1].stem(k, error, basefmt=' ')
axs[1, 1].set_title('Absolute numerical error |DFT - FFT|')
axs[1, 1].set_xlabel('Frequency bin k')
axs[1, 1].set_ylabel('Absolute error')
axs[1, 1].grid(True, alpha=0.3)

fig.suptitle('Simulation 1: Direct DFT and FFT for x = [1, 2, 3, 4]')
fig.savefig('figures/sim_dft_fft.png', dpi=220, bbox_inches='tight')
plt.show()

## Simulation 2 — Approximate operation count

In [ ]:
N = np.array([64, 256, 1024])
dft_count = N**2
fft_count = (N * np.log2(N)).astype(int)
ratio = dft_count / fft_count

operation_table = pd.DataFrame({
    'N': N,
    'DFT approx. N^2': dft_count,
    'FFT approx. N log2(N)': fft_count,
    'DFT / FFT ratio': ratio
})
display(operation_table)

In [ ]:
index = np.arange(len(N))
width = 0.36
fig, axs = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)

axs[0].bar(index - width/2, dft_count, width, label=r'DFT: $N^2$',
           hatch='//', edgecolor='black', facecolor='white')
axs[0].bar(index + width/2, fft_count, width, label=r'FFT: $N\log_2N$',
           hatch='..', edgecolor='black', facecolor='lightgray')
axs[0].set_xticks(index, N)
axs[0].set_xlabel('Transform length N')
axs[0].set_ylabel('Approximate operation count')
axs[0].set_title('Linear scale')
axs[0].legend()
axs[0].grid(True, axis='y', alpha=0.3)

axs[1].plot(N, dft_count, marker='o', label=r'DFT: $N^2$')
axs[1].plot(N, fft_count, marker='s', linestyle='--', label=r'FFT: $N\log_2N$')
axs[1].set_xscale('log', base=2)
axs[1].set_yscale('log')
axs[1].set_xlabel('Transform length N')
axs[1].set_ylabel('Approximate operation count')
axs[1].set_title('Log-log scale')
axs[1].legend()
axs[1].grid(True, which='both', alpha=0.3)

fig.suptitle('Simulation 2: Computational growth of DFT and FFT')
fig.savefig('figures/sim_operation_count.png', dpi=220, bbox_inches='tight')
plt.show()

## Download the figures

Run the following cell in Google Colab to create a ZIP file containing both images.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('fft_simulation_figures', 'zip', 'figures')
files.download('fft_simulation_figures.zip')